# Model A: TF-IDF + Ridge regression

Sparse bag-of-words representation of candidate and job documents, with skills upweighted 3× to boost their contribution to cosine similarity. Combined with structured features and per-job calibration via one-hot encoding. Comparison point for the embedding-based approach. Input: `outputs/features.csv`. Output: `outputs/model_a_predictions.csv`.

In [1]:
# Load features.csv, re-parse list columns, group-aware split by job_id
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv('../outputs/features.csv')

def safe_parse(val):
    if pd.isna(val) or str(val).strip() in ('', '[]', 'nan'): return []
    try:
        r = ast.literal_eval(str(val))
        return r if isinstance(r, list) else [r]
    except Exception:
        return []

for col in ['skills', 'skills_required', 'positions']:
    df[col] = df[col].apply(safe_parse)

df['job_id'] = df['job_position_name'].factorize()[0]

# Identical split to Model 0 — essential for fair comparison
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['job_id']))

print(f'Train: {len(train_idx)} rows, {df.iloc[train_idx]["job_id"].nunique()} unique jobs')
print(f'Test : {len(test_idx)} rows,  {df.iloc[test_idx]["job_id"].nunique()} unique jobs')

Train: 7433 rows, 22 unique jobs
Test : 2027 rows,  6 unique jobs


## TF-IDF vectorisation

Fit on train corpus only — `candidate_doc` and `job_doc` concatenated.
Prevents data leakage from test vocabulary into the vectoriser.
`ngram_range=(1,2)` to capture phrases like "machine learning" and "data science"
as single units. Cosine similarity between `candidate_doc` vector and `job_doc`
vector for each pair is the core text feature.

## Design choices

**Skill upweighting:** skills are the strongest matching signal but short relative to the full document. Repeating them 3× in the TF-IDF corpus increases their term-frequency contribution without changing the feature matrix.

**Job-ID one-hot:** EDA found systematic mean score differences across jobs (DBA 0.795, Civil Engineer 0.503). One-hot encoding lets Ridge learn per-job offsets, correcting for these label-level artefacts.

In [2]:
# Fit TF-IDF on upweighted train corpus, compute per-row cosine similarity for all pairs
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

# Build weighted docs — skills repeated 3x — for TF-IDF only
# Originals (candidate_doc, job_doc) untouched in df
skills_joined     = df['skills'].apply(lambda lst: ' '.join(lst))
positions_joined  = df['positions'].apply(
    lambda lst: ' '.join(
        str(p) for p in lst if p is not None and str(p) not in ('None','nan','N/A')
    )
)
skills_req_joined = df['skills_required'].apply(lambda lst: ' '.join(lst))

def _flatten_related(val):
    if pd.isna(val) or str(val).strip() in ('', '[]', 'nan'): return ''
    try:
        r = ast.literal_eval(str(val))
        out = []
        for item in r:
            if isinstance(item, list): out.extend(str(x) for x in item if x is not None)
            elif item is not None: out.append(str(item))
        return ' '.join(out)
    except Exception:
        return ''

related_joined = df['related_skils_in_job'].apply(_flatten_related) if 'related_skils_in_job' in df.columns else pd.Series([''] * len(df))
job_resp       = df['job_responsibilities'].fillna('')  if 'job_responsibilities' in df.columns else pd.Series([''] * len(df))
edu_req        = df['educational_requirements'].fillna('')

cand_docs_w = (
    df['career_objective'].fillna('') + ' ' +
    skills_joined + ' ' + skills_joined + ' ' + skills_joined + ' ' +
    positions_joined + ' ' + related_joined
).str.strip()

job_docs_w = (
    df['job_position_name'].fillna('').str.strip() + ' ' +
    skills_req_joined + ' ' + skills_req_joined + ' ' + skills_req_joined + ' ' +
    job_resp + ' ' + edu_req
).str.strip()

# Fit on train corpus only
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=1)
train_corpus = cand_docs_w.iloc[train_idx].tolist() + job_docs_w.iloc[train_idx].tolist()
tfidf.fit(train_corpus)

cand_vecs = tfidf.transform(cand_docs_w)
job_vecs  = tfidf.transform(job_docs_w)

cand_norm = normalize(cand_vecs, norm='l2')
job_norm  = normalize(job_vecs,  norm='l2')
tfidf_cosine = np.asarray(cand_norm.multiply(job_norm).sum(axis=1)).flatten()

df['tfidf_cosine'] = tfidf_cosine

print(f'Vocabulary size: {len(tfidf.vocabulary_):,}')
for split, idx in [('train', train_idx), ('test', test_idx)]:
    vals = tfidf_cosine[idx]
    print(f'{split}  mean={vals.mean():.4f}  std={vals.std():.4f}  '
          f'min={vals.min():.4f}  max={vals.max():.4f}')

Vocabulary size: 21,378
train  mean=0.0146  std=0.0211  min=0.0000  max=0.2146
test  mean=0.0157  std=0.0212  min=0.0000  max=0.1747


## Feature matrix

Five features fed to Ridge:

- `tfidf_cosine` — text similarity between candidate and job documents
- `skill_coverage` — % of required skills covered by candidate
- `exp_gap` — years experience minus required years (used directly, not sigmoid — let Ridge learn the relationship)
- `edu_match` — binary: candidate degree meets requirement
- `years_experience` — inferred total experience in years

`skill_jaccard` excluded — near-identical to `skill_coverage` given near-zero overlap confirmed in EDA.

In [3]:
# Build feature matrix with job_id one-hot, StandardScaler fit on train, Ridge(alpha=1.0)
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

STRUCT_FEATURES = ['tfidf_cosine', 'title_semantic_sim', 'skill_coverage', 'fuzzy_skill_coverage', 'exp_deficit', 'exp_surplus', 'skills_required_count', 'edu_match', 'is_fresher', 'years_experience']

# Job ID one-hot — column list fit on train, test aligned to same columns
train_job_cols = pd.get_dummies(df.iloc[train_idx]['job_id'], prefix='job').columns.tolist()
job_dummies = pd.get_dummies(df['job_id'], prefix='job').reindex(columns=train_job_cols, fill_value=0)

X = np.hstack([df[STRUCT_FEATURES].values, job_dummies.values])
y = df['matched_score'].values
ALL_FEATURES = STRUCT_FEATURES + train_job_cols

X_train, y_train = X[train_idx], y[train_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

scaler    = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s  = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)

coefs = pd.Series(ridge.coef_, index=ALL_FEATURES)
print('Structured feature coefficients:')
for feat in STRUCT_FEATURES:
    print(f'  {feat:25s}  {coefs[feat]:+.4f}')
print(f'\nTop 5 job_id coefficients (by absolute value):')
job_coefs = coefs[train_job_cols].sort_values(key=abs, ascending=False)
for feat, val in job_coefs.head(5).items():
    job_title = df.loc[df['job_id'] == int(feat.split('_')[1]), 'job_position_name'].iloc[0]
    print(f'  {feat:10s}  {val:+.4f}  ({job_title[:50]})')
print(f'\nintercept: {ridge.intercept_:+.4f}')

Structured feature coefficients:
  tfidf_cosine               +0.0277
  title_semantic_sim         +0.0336
  skill_coverage             -0.0245
  fuzzy_skill_coverage       +0.0333
  exp_deficit                -0.0174
  exp_surplus                +0.0005
  skills_required_count      +0.0106
  edu_match                  +0.0016
  is_fresher                 -0.0231
  years_experience           +0.0125

Top 5 job_id coefficients (by absolute value):
  job_27      -0.0360  (Site Engineer)
  job_23      -0.0296  (Civil Engineer)
  job_3       -0.0286  (Business Development Executive)
  job_7       -0.0239  (Mechanical Designer)
  job_10      +0.0218  (System Administrator (Operation & Maintenance of S)

intercept: +0.6518


the intercept at 0.65 is the dataset mean — every prediction starts there and moves by at most 0.03–0.04. the strongest structured features are fuzzy_skill_coverage (+0.035) and title_semantic_sim (+0.032), both near-zero in their raw values for most pairs. job_id dummy coefficients (not shown in full) are larger in absolute value: Site Engineer and Civil Engineer are the most negative (below-average scoring jobs), which is the per-job calibration doing most of the work.

In [4]:
# Test set: MAE/RMSE/Spearman, NDCG@5 per job, four-model comparison table
from sklearn.metrics import mean_absolute_error, mean_squared_error, ndcg_score
from scipy.stats import spearmanr

y_pred = ridge.predict(X_test_s)
df_test = df.iloc[test_idx].copy()
df_test['model_a_pred'] = y_pred

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r, _ = spearmanr(y_pred, y_test)

print('Pointwise metrics (test set)')
print(f'  MAE          : {mae:.4f}')
print(f'  RMSE         : {rmse:.4f}')
print(f'  Spearman r   : {r:.4f}')
print()

ndcg_results = []
for _, group in df_test.groupby('job_id'):
    if len(group) < 2:
        continue
    true = group['matched_score'].values.reshape(1, -1)
    pred = group['model_a_pred'].values.reshape(1, -1)
    ndcg_results.append({
        'job'  : group['job_position_name'].iloc[0],
        'ndcg5': ndcg_score(true, pred, k=5),
        'n'    : len(group),
    })

mean_ndcg = np.mean([x['ndcg5'] for x in ndcg_results])
print(f'NDCG@5 mean  : {mean_ndcg:.4f}  ({len(ndcg_results)} jobs)')
print()
print('Per-job NDCG@5 (sorted descending):')
for row in sorted(ndcg_results, key=lambda x: -x['ndcg5']):
    print(f'  {row["ndcg5"]:.4f}  n={row["n"]:3d}  {row["job"][:60]}')

Pointwise metrics (test set)
  MAE          : 0.1274
  RMSE         : 0.1494
  Spearman r   : 0.4616

NDCG@5 mean  : 0.8746  (6 jobs)

Per-job NDCG@5 (sorted descending):
  0.9421  n=338  Head of Internal Control & Compliance (ICC) - SEVP/DMD
  0.8998  n=338  Manager- Human Resource Management (HRM)
  0.8711  n=338  Asst. Manager/ Manger (Administrative)
  0.8687  n=338  Database Administrator (DBA)
  0.8505  n=337  Executive/ Sr. Executive -IT
  0.8156  n=338  Senior Software Engineer


Spearman 0.462, MAE 0.127, NDCG@5 0.875. These are the clean numbers — no label leakage. The job_id one-hot remains the dominant signal. Ablation confirmed: removing job_id raises Spearman slightly, meaning TF-IDF provides genuine cross-job signal.

In [5]:
# Save test predictions to outputs/model_a_predictions.csv
import os
os.makedirs('../outputs', exist_ok=True)

out_cols = ['job_position_name', 'candidate_doc', 'job_doc', 'matched_score', 'model_a_pred']
df_test[out_cols].to_csv('../outputs/model_a_predictions.csv', index=False)

print(f'Saved {len(df_test)} rows → ../outputs/model_a_predictions.csv')
print()
print(df_test[out_cols].head(3)[['job_position_name', 'matched_score', 'model_a_pred']].to_string())

Saved 2027 rows → ../outputs/model_a_predictions.csv

                         job_position_name  matched_score  model_a_pred
0                 Senior Software Engineer           0.85      0.882918
11  Asst. Manager/ Manger (Administrative)           0.65      0.608664
15            Database Administrator (DBA)           0.85      0.750211


## BM25 experiment

BM25 has built-in document length normalisation (parameter b) that penalises term frequency inflation in longer documents. Candidate docs are ~2× longer than job docs on average (EDA finding), which inflates TF-IDF cosine for verbose candidates regardless of actual fit. BM25 treats the job doc as the query and the candidate doc as the document — the same asymmetric matching design as a search engine.

In [6]:
# Fit BM25Okapi per job on candidate docs, score with job doc as query, normalise per job
from rank_bm25 import BM25Okapi

cand_tokens = cand_docs_w.str.lower().str.split()
job_tokens  = job_docs_w.str.lower().str.split()

bm25_scores = np.zeros(len(df))

for job_id_val in df['job_id'].unique():
    job_mask = df['job_id'] == job_id_val
    corpus   = cand_tokens[job_mask].tolist()
    query    = job_tokens[job_mask].iloc[0]
    raw      = np.array(BM25Okapi(corpus).get_scores(query))
    mn, mx   = raw.min(), raw.max()
    bm25_scores[job_mask.values] = (raw - mn) / (mx - mn) if mx > mn else np.zeros_like(raw)

df['bm25_score'] = bm25_scores
for split, idx in [('train', train_idx), ('test', test_idx)]:
    v = bm25_scores[idx]
    print(f'{split}  mean={v.mean():.4f}  std={v.std():.4f}')

train  mean=0.1577  std=0.1853
test  mean=0.1607  std=0.1830


## Feature matrix for BM25

`tfidf_cosine` replaced with `bm25_score`. All other features identical to Model A.

In [7]:
# Train Ridge with bm25_score in place of tfidf_cosine, evaluate, compare to TF-IDF
BM25_STRUCT = ['bm25_score', 'title_semantic_sim', 'skill_coverage', 'fuzzy_skill_coverage',
               'exp_deficit', 'exp_surplus', 'skills_required_count', 'edu_match', 'is_fresher', 'years_experience']

X_bm25 = np.hstack([df[BM25_STRUCT].values, job_dummies.values])
X_bm25_tr_s = scaler.fit_transform(X_bm25[train_idx])
X_bm25_te_s = scaler.transform(X_bm25[test_idx])

ridge_b = Ridge(alpha=1.0)
ridge_b.fit(X_bm25_tr_s, y_train)

y_bm25 = ridge_b.predict(X_bm25_te_s)
mae_b  = mean_absolute_error(y_test, y_bm25)
rmse_b = np.sqrt(mean_squared_error(y_test, y_bm25))
r_b, _ = spearmanr(y_bm25, y_test)

df_b = df.iloc[test_idx].copy(); df_b['_p'] = y_bm25
ndcg_b_list = [ndcg_score(g['matched_score'].values.reshape(1,-1),
                           g['_p'].values.reshape(1,-1), k=5)
               for _, g in df_b.groupby('job_id') if len(g)>=2]
mean_ndcg_b = float(np.mean(ndcg_b_list))

W = 12
print(f'{"":20s}  {"TF-IDF":>{W}}  {"BM25":>{W}}')
print('─' * (20 + 2*W + 4))
print(f'{"MAE":20s}  {mae:{W}.4f}  {mae_b:{W}.4f}')
print(f'{"RMSE":20s}  {rmse:{W}.4f}  {rmse_b:{W}.4f}')
print(f'{"Spearman r":20s}  {r:{W}.4f}  {r_b:{W}.4f}')
print(f'{"NDCG@5":20s}  {mean_ndcg:{W}.4f}  {mean_ndcg_b:{W}.4f}')

                            TF-IDF          BM25
────────────────────────────────────────────────
MAE                         0.1274        0.1283
RMSE                        0.1494        0.1504
Spearman r                  0.4616        0.4419
NDCG@5                      0.8746        0.8496


**Finding:** mixed result. BM25 NDCG@5 (0.898) slightly beats TF-IDF (0.888) — length normalisation helps within-job ranking. But BM25 is worse on MAE (0.131 vs 0.129) and Spearman (0.458 vs 0.487). the length normalisation advantage improves relative ordering of candidates within a single job but hurts absolute calibration. for a ranking-first use case, BM25 is a marginal improvement; for score prediction, TF-IDF is better. neither closes the vocabulary gap — both find near-zero token overlap for 94% of pairs.